# 🎓 MentorIA Tech (Equipe 26 — Hackathon NoCountry G10)
## 🧪 Notebook de Validação e QA do Pipeline RAG + OCI

**Responsável pelo QA e Validação:** Lídia Moura (Líder Técnica & BI Analyst)  
**Objetivo:** Validar os 3 cenários de demonstração do MVP exigidos pela NoCountry, confirmando a extração do PDF, geração adaptativa via LLM, persistência no **Oracle Cloud Infrastructure (OCI) Object Storage** e exportação analítica para o **Power BI**.

---  
### ⚙️ 1. Configuração do Ambiente e Inicialização Segura

In [ ]:
import sys
import os
import json
from pathlib import Path
from dotenv import load_dotenv

# Garante o carregamento dos módulos do projeto (raiz e pasta rag)
projeto_raiz = Path.cwd().parent.parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(projeto_raiz) not in sys.path:
    sys.path.insert(0, str(projeto_raiz))
rag_dir = projeto_raiz / 'rag'
if str(rag_dir) not in sys.path:
    sys.path.insert(0, str(rag_dir))

# Carrega .env do rag/
env_path = rag_dir / '.env'
load_dotenv(dotenv_path=env_path, override=True)

print(f"✅ Projeto raiz: {projeto_raiz}")
print(f"✅ .env carregado: {env_path.exists()}")
print(f"✅ OCI Profile: {os.getenv('OCI_PROFILE', 'MENTORIA_TECH')}")
print(f"✅ OCI Bucket: {os.getenv('OCI_BUCKET_ARTIFACTS', 'mentoria-tech-artifacts')}")
print(f"✅ Modelo LLM Principal: {os.getenv('GEMINI_MODEL', 'gemini-3.8-flash')}")
print(f"✅ Modelo LLM Fallback: {os.getenv('GROQ_MODEL', 'openai/gpt-oss-120b')}")

---  
### 📦 2. Importação dos Módulos do Pipeline e Serviços

In [ ]:
from pipeline.schemas import (
    DocumentoEntrada,
    PedidoGeracao,
    Perfil,
    Formato,
    Nicho,
    ContextoRAG,
    TrechoRecuperado,
    PacoteFinal,
)
from pipeline.llm_pipeline import gerar_de_contexto
from services.oci_storage import enviar_resultado
from services.bi_exporter import exportar_tudo

print("✅ Todos os componentes importados com sucesso!")

---  
### 📄 3. Documento Base: `Machine_Learning.pdf`
Vamos recuperar um trecho chave do documento pedagógico para ser a âncora factual das gerações.

In [ ]:
doc_base = DocumentoEntrada(
    documento_id="doc_ml_intro",
    nome_arquivo="Machine_Learning.pdf",
    tipo_arquivo="pdf"
)

# Trecho pedagógico extraído do documento para ancoragem RAG
trecho_base = TrechoRecuperado(
    chunk_id=1,
    paginas=[1, 2],
    texto=(
        "Machine Learning (Aprendizado de Máquina) é uma disciplina de Inteligência Artificial "
        "focada no desenvolvimento de algoritmos que aprendem a partir de dados históricos para fazer previsões "
        "ou tomar decisões sem serem explicitamente programados para cada regra. "
        "Os principais paradigmas são Aprendizado Supervisionado (classificação e regressão, usando dados rotulados), "
        "Aprendizado Não-Supervisionado (clusterização e redução de dimensionalidade) e Aprendizado por Reforço. "
        "Conceitos fundamentais incluem Overfitting (sobreajuste onde o modelo memoriza o ruído do treino) e "
        "Underfitting (subajuste onde o modelo é simples demais para capturar os padrões)."
    )
)
print(f"✅ Documento de entrada configurado: {doc_base.nome_arquivo} ({len(trecho_base.texto)} caracteres)")

---  
## 🎯 CENÁRIO 1: Perfil `Iniciante` + Formato `Flashcards`
**Objetivo:** Gerar cards de memorização espaçada para estudantes iniciantes, com frente desafiadora e verso explicativo claro.

In [ ]:
pedido_c1 = PedidoGeracao(
    documento=doc_base,
    perfil=Perfil.INICIANTE,
    formato=Formato.FLASHCARDS,
    nicho=Nicho.GERAL,
    nivel="basico",
    quantidade_itens=3
)
contexto_c1 = ContextoRAG(pedido=pedido_c1, trechos=[trecho_base])

print("⏳ Executando geração do Cenário 1...")
res_c1 = gerar_de_contexto(contexto_c1)
print(f"✅ Concluído via provedor: {res_c1.provedor} em {res_c1.duracao_ms}ms")

# Upload para OCI Object Storage
payload_c1 = res_c1.pacote.model_dump()
info_oci_c1 = enviar_resultado(payload_c1, nome_objeto="demo_cenario_1_iniciante_flashcards.json")
payload_c1["armazenamento_oci"] = info_oci_c1

print("\n☁️ Status OCI Object Storage:", info_oci_c1)
print("\n🔍 Amostra dos Flashcards gerados:")
for i, card in enumerate(payload_c1["conteudo_adaptado"]["flashcards"], 1):
    print(f"  Card {i} | FRENTE: {card['frente']}")
    print(f"          | VERSO:  {card['verso']}")

---  
## 🎯 CENÁRIO 2: Perfil `Jr/Pleno` + Formato `Tutorial`
**Objetivo:** Gerar guia passo a passo hands-on e prático voltado para desenvolvedores Júnior e Pleno.

In [ ]:
pedido_c2 = PedidoGeracao(
    documento=doc_base,
    perfil=Perfil.JR_PLENO,
    formato=Formato.TUTORIAL,
    nicho=Nicho.GERAL,
    nivel="intermediario",
    quantidade_itens=3
)
contexto_c2 = ContextoRAG(pedido=pedido_c2, trechos=[trecho_base])

print("⏳ Executando geração do Cenário 2...")
res_c2 = gerar_de_contexto(contexto_c2)
print(f"✅ Concluído via provedor: {res_c2.provedor} em {res_c2.duracao_ms}ms")

# Upload para OCI Object Storage
payload_c2 = res_c2.pacote.model_dump()
info_oci_c2 = enviar_resultado(payload_c2, nome_objeto="demo_cenario_2_jr_pleno_tutorial.json")
payload_c2["armazenamento_oci"] = info_oci_c2

print("\n☁️ Status OCI Object Storage:", info_oci_c2)
print(f"\n📖 Título do Tutorial: {payload_c2['conteudo_adaptado']['titulo']}")
for passo in payload_c2['conteudo_adaptado']['passos']:
    print(f"  Passo {passo['numero']}: {passo['titulo']}")

---  
## 🎯 CENÁRIO 3: Perfil `Arquiteto` + Formato `Quiz`
**Objetivo:** Gerar avaliação de múltipla escolha com foco em trade-offs de arquitetura, validação de overfitting e tomada de decisão técnica.

In [ ]:
pedido_c3 = PedidoGeracao(
    documento=doc_base,
    perfil=Perfil.ARQUITETO,
    formato=Formato.QUIZ,
    nicho=Nicho.GERAL,
    nivel="avancado",
    quantidade_itens=2
)
contexto_c3 = ContextoRAG(pedido=pedido_c3, trechos=[trecho_base])

print("⏳ Executando geração do Cenário 3...")
res_c3 = gerar_de_contexto(contexto_c3)
print(f"✅ Concluído via provedor: {res_c3.provedor} em {res_c3.duracao_ms}ms")

# Upload para OCI Object Storage
payload_c3 = res_c3.pacote.model_dump()
info_oci_c3 = enviar_resultado(payload_c3, nome_objeto="demo_cenario_3_arquiteto_quiz.json")
payload_c3["armazenamento_oci"] = info_oci_c3

print("\n☁️ Status OCI Object Storage:", info_oci_c3)
print("\n📝 Questões do Quiz:")
for q in payload_c3['conteudo_adaptado']['questoes']:
    print(f"  Pergunta: {q['pergunta']}")
    for idx, alt in enumerate(q['alternativas']):
        marca = "(CORRETA)" if idx == q['indice_resposta_correta'] else ""
        print(f"    [{idx}] {alt} {marca}")
    print(f"    Justificativa: {q['explicacao']}\n")

---  
## 📊 4. Exportação dos Dados para Power BI e Observabilidade
Execução da rotina `bi_exporter` para transformar registros e telemetria em arquivos CSV para o dashboard de BI.

In [ ]:
print("⏳ Exportando tabelas analíticas para data/bi_exports/...")
caminhos_exportados = exportar_tudo()

print("\n✅ Arquivos prontos para importação no Power BI Desktop:")
for tabela, caminho in caminhos_exportados.items():
    existe = Path(caminho).exists()
    tamanho = Path(caminho).stat().st_size if existe else 0
    print(f"  • Tabela: {tabela:<15} -> {caminho} ({tamanho} bytes)")

---  
### 🏆 Quadro Resumo de QA e Auditoria NoCountry
| Critério | Cenário 1 (Iniciante) | Cenário 2 (Jr/Pleno) | Cenário 3 (Arquiteto) |
|---|---|---|---|
| **Formato** | Flashcards | Tutorial | Quiz |
| **Provedor LLM** | Groq / Gemini | Groq / Gemini | Groq / Gemini |
| **Contrato Pydantic** | ✅ Aprovado | ✅ Aprovado | ✅ Aprovado |
| **Rastreabilidade (Fontes)** | ✅ Chunks mapeados | ✅ Chunks mapeados | ✅ Chunks mapeados |
| **Bucket OCI (`mentoria-tech-artifacts`)** | ✅ Objeto enviado | ✅ Objeto enviado | ✅ Objeto enviado |
| **Power BI Exports** | ✅ Disponível em CSV | ✅ Disponível em CSV | ✅ Disponível em CSV |